# A1 Fatal Traffic Accidents — Geospatial Analysis
**Source**: stg_accidents (PostgreSQL)  
**Goal**: 以經緯度視覺化全台死亡事故熱點分布

In [1]:
import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster
from sqlalchemy import create_engine

engine = create_engine(
    'postgresql://postgres@localhost:5432/postgres',
    connect_args={'options': '-csearch_path=A1_Traffic_Fatal'}
)
print('Connected.')

Connected.


## 1. 載入座標資料

In [2]:
df = pd.read_sql('SELECT * FROM "A1_Traffic_Fatal".stg_accidents', engine)

# 以事故為單位，取 party_order=1，排除無座標
acc = df[df['party_order'] == 1].copy()
acc = acc.dropna(subset=['latitude', 'longitude'])
acc = acc[(acc['latitude'] != 0) & (acc['longitude'] != 0)]

print(f'有效事故筆數：{len(acc)}')
acc[['latitude', 'longitude']].describe()

有效事故筆數：416


,latitude,longitude
count,416.000000,416.000000
mean,23.992379,120.817154
std,0.860020,0.490757
min,22.010757,119.600204
25%,23.214578,120.395769
50%,24.007544,120.675883
75%,24.918843,121.285314
max,25.284937,121.914960


## 2. 全台事故熱點地圖 (HeatMap)

In [3]:
# 台灣中心點
taiwan_center = [23.5, 121.0]

m = folium.Map(location=taiwan_center, zoom_start=8, tiles='CartoDB positron')

heat_data = acc[['latitude', 'longitude']].values.tolist()
HeatMap(
    heat_data,
    radius=15,
    blur=10,
    min_opacity=0.4,
    gradient={0.4: 'blue', 0.6: 'lime', 0.8: 'orange', 1.0: 'red'}
).add_to(m)

m.save('../Images/heatmap_taiwan.html')
print('儲存完成：Images/heatmap_taiwan.html')
m

儲存完成：Images/heatmap_taiwan.html


## 3. 分時段熱點地圖 (時段對比)

In [4]:
def time_period(h):
    if 6 <= h <= 9:    return '早尖峰'
    elif 10 <= h <= 15: return '日間'
    elif 16 <= h <= 19: return '晚尖峰'
    elif 20 <= h <= 23: return '夜間'
    else:               return '深夜凌晨'

acc['time_period'] = acc['accident_hour'].apply(time_period)

period_colors = {
    '早尖峰': 'orange',
    '日間':   'blue',
    '晚尖峰': 'red',
    '夜間':   'purple',
    '深夜凌晨': 'black'
}

m2 = folium.Map(location=taiwan_center, zoom_start=8, tiles='CartoDB positron')

for period, color in period_colors.items():
    sub = acc[acc['time_period'] == period]
    fg = folium.FeatureGroup(name=f'{period} ({len(sub)}起)')
    for _, row in sub.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=4,
            color=color,
            fill=True,
            fill_opacity=0.6,
            popup=folium.Popup(
                f"<b>{row['accident_date']}</b><br>"
                f"時段：{period}<br>"
                f"肇因：{row['cause_minor']}<br>"
                f"天候：{row['weather']}<br>"
                f"死亡：{int(row['death_count'])}人",
                max_width=200
            )
        ).add_to(fg)
    fg.add_to(m2)

folium.LayerControl().add_to(m2)
m2.save('../Images/map_by_period.html')
print('儲存完成：Images/map_by_period.html')
m2

儲存完成：Images/map_by_period.html


## 4. 肇逃事故地圖

In [5]:
m3 = folium.Map(location=taiwan_center, zoom_start=8, tiles='CartoDB positron')

for _, row in acc.iterrows():
    color = 'red' if row['is_hit_and_run'] else 'steelblue'
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5 if row['is_hit_and_run'] else 3,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>{'肇逃' if row['is_hit_and_run'] else '一般'}</b><br>"
            f"{row['accident_date']}<br>"
            f"地點：{row['location']}",
            max_width=200
        )
    ).add_to(m3)

m3.save('../Images/map_hit_and_run.html')
print('儲存完成：Images/map_hit_and_run.html')
m3

儲存完成：Images/map_hit_and_run.html
